# Lab 06-04 — The repo's rerankers head-to-head (nDCG)

**Track 06 · Re-ranking** — the reranker arsenal wired end-to-end onto the same nfcorpus pool as lab 02, scored head-to-head against one bi-encoder baseline.

Labs 01-03 of this track used the cross-encoder reranker directly. The component library ships more: `src/retrieval/rerank.py` wraps the same engine in a composable `RerankRetriever` — retrieve wide, rerank short, one call — and `src/retrieval/rerank_advanced.py` adds three more families: ColBERT's token-level MaxSim late interaction, a pointwise MonoT5, and LLM pointwise scoring. This notebook is **self-contained**: every one of those rerankers is built inline, right here, with the LangChain ecosystem + raw libraries — no repo component imports. The inline classes mirror the shared components arithmetic-for-arithmetic, so the notebook measures exactly what the lab measures.

This lab wires them all onto the same nfcorpus pool as lab 02 and measures each on nDCG@5 against the bi-encoder baseline:

* `RerankRetriever` — bi top-20, cross-encoder top-5. The production shape.
* `ColBERTReranker` — bi top-20, MaxSim over per-token BGE embeddings, top-5. No extra model download: it reuses the same BGE embedder as retrieval.
* `MonoT5` — pointwise seq2seq "is this relevant?" — OPTIONAL, only runs if the ~3GB checkpoint is already in the local HF cache (skipped otherwise).
* `LLMPointwise` — "yes/no" per candidate via Groq — OPTIONAL, only runs with `--with-llm` (uses `GROQ_API_KEY`; O(n) calls, keep the pool tiny).

```text
600 nfcorpus docs
  -> BGE embeddings (local, cpu) -> FAISS index
  -> per query: bi top-20 candidates
  -> rerankers { RerankRetriever | ColBERT | MonoT5* | LLM pointwise* }
  -> nDCG@5 vs the bi-encoder baseline
  -> verification gate (--verify)      * = optional
```

The verification gate covers only the two local systems; the optional sections print a SKIP notice when their dependency is missing. This notebook runs the lab's default (`with_llm=False`), so the Groq LLM reranker is skipped — `exp` still records its status so the gate stays honest.


## Setup

One prerequisite must hold before this notebook will run:

- **nfcorpus on disk** — `Data/corpus/beir-nfcorpus/nfcorpus/` (BEIR medical FAQ corpus: `corpus.jsonl` + `queries.jsonl` + `qrels/test.tsv`), already fetched by the repo's manifest-verified fetchers.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-huggingface`, `langchain-community`, `langchain-classic`, `sentence-transformers`, and `faiss-cpu`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
#   groq -> optional LLMPointwiseReranker (--with-llm only)
%pip install -q sentence-transformers langchain-huggingface langchain-community langchain-classic faiss-cpu


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import json
import math
import os
import sys
import time
from pathlib import Path

# LangChain + sentence-transformers + faiss — the only libraries this
# notebook needs. Nothing is imported from the repo's src/ component library.
import numpy as np  # noqa: E402
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.cross_encoders import BaseCrossEncoder  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402
from sentence_transformers import CrossEncoder  # noqa: E402
from sentence_transformers import SentenceTransformer  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `NF_N_DOCS = 600` indexes the deterministic head of the 3633-doc nfcorpus corpus (enough for stable rankings at a fraction of the embed time), `NF_MAX_QUERIES = 40` fixes the pool at the first 40 qrels-covered queries with gold inside the subset, and `WIDE_K = 20` / `EVAL_K = 5` fix the stage-1 bi-encoder candidate list and the ranking depth every system is scored at. `BGE_MODEL_NAME` picks the same BGE embedder the whole repo uses by default, `CE_MODEL_NAME` the local cross-encoder, `LLM_POOL = 3` keeps the optional LLM reranker's O(n) Groq calls tiny, and `MONT5_CACHE` points at the ~3GB MonoT5 checkpoint the lab only runs when it is already cached.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
NFCORPUS_DIR = Path("Data/corpus/beir-nfcorpus/nfcorpus")
CORPUS_PATH = NFCORPUS_DIR / "corpus.jsonl"
QUERIES_PATH = NFCORPUS_DIR / "queries.jsonl"
QRELS_PATH = NFCORPUS_DIR / "qrels" / "test.tsv"
NF_N_DOCS = 600  # deterministic head of the 3633-doc nfcorpus corpus
NF_MAX_QUERIES = 40  # pool: first 40 qrels-covered queries with gold inside
WIDE_K = 20  # stage-1 bi-encoder candidate list
EVAL_K = 5  # ranking depth every system is scored at
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
CE_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_POOL = 3  # LLM pointwise demo: first 3 pool queries, top-10 candidates
MONT5_CACHE = (
    Path.home() / ".cache/huggingface/hub/models--castorini--monot5-base-msmarco"
)


## 2. Load — nfcorpus corpus + queries + qrels

Three plain-text loaders, one per file: `load_nfcorpus` reads the first `n` corpus lines and concatenates each doc's `title` and `text` into one indexed string (the id is kept alongside), `load_nf_queries` keeps every query in file order, and `load_qrels` parses the BEIR qrels TSV keeping only pairs with score >= 1. The corpus and queries stay in their JSONL shape — each whole doc is one unit, exactly as lab 02 prepared the pool.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — nfcorpus corpus + queries + qrels
# --------------------------------------------------------------------------
def load_nfcorpus(path: Path, n: int) -> tuple[list[str], list[str]]:
    """Return (doc_texts, doc_ids) for the first ``n`` corpus docs."""
    texts: list[str] = []
    ids: list[str] = []
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            doc = json.loads(line)
            ids.append(doc["_id"])
            texts.append(f'{doc["title"]} {doc["text"]}')
    return texts, ids


def load_nf_queries(path: Path) -> list[tuple[str, str]]:
    """Return [(query_id, query_text)] for every query, in file order."""
    out: list[tuple[str, str]] = []
    with open(path) as f:
        for line in f:
            q = json.loads(line)
            out.append((q["_id"], q["text"]))
    return out


def load_qrels(path: Path) -> dict[str, set[str]]:
    """Return {query_id: {relevant_corpus_id, ...}} (qrels score >= 1)."""
    qrels: dict[str, set[str]] = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 3 or parts[0] == "query-id":
                continue
            qid, cid = parts[0], parts[1]
            if int(parts[2]) >= 1:
                qrels.setdefault(qid, set()).add(cid)
    return qrels


## 3. Metric — nDCG@k (same implementation as lab 02)

`ndcg_at_k` is binary-relevance nDCG with the standard log2 gain discount, identical to lab 02 — so the numbers in this lab are directly comparable to that one. `ranked_ids[:k]` caps the depth, `min(k, len(gold))` builds the ideal DCG from the relevant set, and an empty gold yields 0.0 instead of a division by zero.


In [ ]:
# --------------------------------------------------------------------------
# 3. Metric — nDCG@k (same implementation as lab 02)
# --------------------------------------------------------------------------
def ndcg_at_k(ranked_ids: list[str], gold: set[str], k: int) -> float:
    """nDCG@k in [0, 1] with binary relevance."""
    gain = 0.0
    for i, cid in enumerate(ranked_ids[:k], start=1):
        if cid in gold:
            gain += 1.0 / math.log2(i + 1)
    ideal = 0.0
    for i in range(1, min(k, len(gold)) + 1):
        ideal += 1.0 / math.log2(i + 1)
    return gain / ideal if ideal > 0.0 else 0.0


## 4. Experiment — index, then score each reranker on the same pool

The whole lab in one function, with every reranker built inline as a plain class — each mirrors its shared-component counterpart in `src/retrieval/rerank.py` and `src/retrieval/rerank_advanced.py` arithmetic-for-arithmetic:

* `_InlineRerankRetriever` — retrieve wide with the bi-encoder store, re-score the `k_retrieve` candidates with the sentence-transformers `CrossEncoder`, keep `top_k` (the `RerankRetriever` shape);
* `_InlineColBERT` — per-token BGE embeddings via `SentenceTransformer.encode(output_value="token_embeddings")`, then numpy MaxSim (sum over query tokens of their best doc-token cosine);
* `_InlineMonoT5` — a `transformers` text2text pipeline scoring `true_logit - false_logit` at the first decoding step;
* `_InlineLLMPointwise` — one yes/no LLM call per candidate, gated behind `with_llm` + `GROQ_API_KEY`.

`run_experiment` embeds the 600 docs once with BGE, builds the FAISS store (precomputed vectors for the index step, real query embeds for the retriever wrappers), and scores the same 40-query pool at nDCG@5 for the bi-encoder baseline, the RerankRetriever shape, and ColBERT — then the classic LangChain wiring (`ContextualCompressionRetriever` + `CrossEncoderReranker` from `langchain-classic`, adapted through a `BaseCrossEncoder`) on the same pool as a cross-check, then the two optional systems, each printing a SKIP notice when its dependency is missing. The returned `exp` dict is the artifact both the demo and the gate inspect.


In [ ]:
# --------------------------------------------------------------------------
# 4. Experiment — index, then score each reranker on the same pool
# --------------------------------------------------------------------------
class _MixedEmbeddings(Embeddings):
    """Precomputed for embed_documents (already timed), real embed for queries.

    The store is built from the precomputed 600-doc vectors so the embed step
    and the index step stay separately timed; embed_query delegates to the
    real BGE embedder so the store can also serve as a BaseRetriever for the
    classic ContextualCompressionRetriever wiring below.
    """

    def __init__(self, texts: list[str], embeddings: list[list[float]],
                 embedder: Embeddings):
        if len(texts) != len(embeddings):
            raise ValueError("texts and embeddings must be parallel lists")
        self._table: dict[str, list[float]] = dict(zip(texts, embeddings))
        self._embedder = embedder

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        missing = [t for t in texts if t not in self._table]
        if missing:
            raise ValueError(f"{len(missing)} text(s) have no precomputed vector")
        return [self._table[t] for t in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embedder.embed_query(text)


class _InlineRerankRetriever:
    """Retrieve wide with the bi-encoder, rerank short with the cross-encoder.

    Mirrors src/retrieval/rerank.py::RerankRetriever: one call pulls
    ``k_retrieve`` candidates from the store, then re-scores them with the
    inline cross-encoder down to ``top_k``.
    """

    def __init__(self, store, embedder, ce_model, k_retrieve: int = WIDE_K,
                 top_k: int = EVAL_K):
        self.store = store
        self.embedder = embedder
        self.ce_model = ce_model
        self.k_retrieve = k_retrieve
        self.top_k = top_k

    def retrieve(self, question: str) -> list[Document]:
        candidates = self.store.similarity_search_by_vector(
            self.embedder.embed_query(question), k=self.k_retrieve
        )
        if not candidates:
            return []
        pairs = [(question, d.page_content) for d in candidates]
        scores = self.ce_model.predict(pairs, show_progress_bar=False, batch_size=32)
        ranked = sorted(zip(scores, candidates), key=lambda p: p[0], reverse=True)
        return [d for _, d in ranked[: self.top_k]]


class _InlineColBERT:
    """ColBERT-style MaxSim reranker, built inline.

    Mirrors src/retrieval/rerank_advanced.py::ColBERTReranker: per-token BGE
    embeddings, then for every query token its best-matching document token
    (cosine), summed. No extra model download — the same BGE embedder as
    retrieval.
    """

    def __init__(self, model_name: str = BGE_MODEL_NAME):
        self._model = SentenceTransformer(model_name)

    def _token_embeddings(self, text: str) -> list[list[float]]:
        emb = self._model.encode(text, output_value="token_embeddings")
        return emb.tolist() if hasattr(emb, "tolist") else list(emb)

    def _maxsim_score(self, query_tokens: list[list[float]],
                      doc_tokens: list[list[float]]) -> float:
        if not query_tokens or not doc_tokens:
            return 0.0
        q = np.asarray(query_tokens, dtype=np.float32)
        d = np.asarray(doc_tokens, dtype=np.float32)
        q = q / np.linalg.norm(q, axis=1, keepdims=True)
        d = d / np.linalg.norm(d, axis=1, keepdims=True)
        sims = q @ d.T  # (n_query_tokens, n_doc_tokens) cosine matrix
        return float(sims.max(axis=1).sum())

    def rerank(self, query: str, documents: list[Document],
               top_k: int = EVAL_K) -> list[Document]:
        if not documents:
            return []
        q_tokens = self._token_embeddings(query)
        scored: list[tuple[float, Document]] = []
        for doc in documents:
            d_tokens = self._token_embeddings(doc.page_content)
            score = self._maxsim_score(q_tokens, d_tokens)
            doc.metadata["score"] = score
            scored.append((score, doc))
        scored.sort(key=lambda pair: pair[0], reverse=True)
        return [doc for _, doc in scored[:top_k]]


class _InlineMonoT5:
    """Pointwise seq2seq reranker via a transformers pipeline (MonoT5).

    Mirrors src/retrieval/rerank_advanced.py::MonoT5Reranker: the score is
    the logit difference of the ``true``/``false`` tokens at the first
    decoding step of the ``castorini/monot5-base-msmarco`` checkpoint.
    """

    DEFAULT_MODEL = "castorini/monot5-base-msmarco"

    def __init__(self, model_name: str = DEFAULT_MODEL):
        self.model_name = model_name
        self._pipeline = None

    def _get_pipeline(self):
        if self._pipeline is None:
            from transformers import pipeline
            self._pipeline = pipeline(
                "text2text-generation", model=self.model_name, max_length=512
            )
        return self._pipeline

    def _score_pair(self, query: str, passage: str) -> float:
        prompt = f"Query: {query} Document: {passage} Relevant:"
        result = self._get_pipeline()(
            prompt,
            max_length=512,
            return_dict_in_generate=True,
            output_scores=True,
        )
        out = result[0] if isinstance(result, (list, tuple)) else result
        tokenizer = self._get_pipeline().tokenizer
        true_id = tokenizer.convert_tokens_to_ids("true")
        false_id = tokenizer.convert_tokens_to_ids("false")
        logits = out["scores"][0][0]  # first decoding step, batch item 0
        return float(logits[true_id] - logits[false_id])

    def rerank(self, query: str, documents: list[Document],
               top_k: int = EVAL_K) -> list[Document]:
        if not documents:
            return []
        scored: list[tuple[float, Document]] = []
        for doc in documents:
            score = self._score_pair(query, doc.page_content)
            doc.metadata["score"] = score
            scored.append((score, doc))
        scored.sort(key=lambda pair: pair[0], reverse=True)
        return [doc for _, doc in scored[:top_k]]


class _InlineLLMPointwise:
    """Rerank by asking an LLM "is this relevant? yes/no" per candidate.

    Mirrors src/retrieval/rerank_advanced.py::LLMPointwiseReranker: pointwise
    (each candidate judged independently), so O(n) LLM calls — keep the pool
    tiny. The LLM is any object exposing ``invoke(prompt) -> str``.
    """

    YES_NO_PROMPT = """You are a reranker. Given a query and one candidate
passage, answer ONLY "yes" or "no" — is the passage relevant to answering
the query?

Query: {query}

Passage: {passage}

Relevant:"""

    def __init__(self, llm):
        self.llm = llm

    def _yes_no(self, query: str, passage: str) -> bool:
        response = self.llm.invoke(
            self.YES_NO_PROMPT.format(query=query, passage=passage)
        )
        answer = response.strip().lower().rstrip(".,!? \n")
        return (
            answer in ("yes", "y", "true", "relevant")
            or answer.startswith("yes ")
        )

    def rerank(self, query: str, documents: list[Document],
               top_k: int = 10) -> list[Document]:
        relevant = [d for d in documents if self._yes_no(query, d.page_content)]
        return relevant[:top_k]


class _GroqTextLLM:
    """Adapter: ChatGroq.invoke returns an AIMessage; the pointwise reranker
    wants invoke(prompt) -> str. Mirrors src/llms/groq.py::GroqLLM."""

    def __init__(self, model: str = "llama-3.3-70b-versatile", temperature: float = 0.0):
        from langchain_groq import ChatGroq
        self._llm = ChatGroq(model=model, temperature=temperature)

    def invoke(self, prompt: str) -> str:
        response = self._llm.invoke(prompt)
        return response.content if hasattr(response, "content") else str(response)


class _ScoringCrossEncoder(BaseCrossEncoder):
    """Adapter: sentence-transformers CrossEncoder -> langchain_core
    BaseCrossEncoder (the classic CrossEncoderReranker's model contract)."""

    def __init__(self, model_name: str = CE_MODEL_NAME):
        self._ce = CrossEncoder(model_name)

    def score(self, text_pairs: list[tuple[str, str]]) -> list[float]:
        return [float(s) for s in self._ce.predict(text_pairs, show_progress_bar=False)]


def run_experiment(with_llm: bool = False) -> dict:
    nf_texts, nf_ids = load_nfcorpus(CORPUS_PATH, NF_N_DOCS)
    nf_queries = load_nf_queries(QUERIES_PATH)
    qrels = load_qrels(QRELS_PATH)

    # --- Embed locally (BGE) and index in-memory ---------------------------
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )
    t0 = time.perf_counter()
    nf_vecs = embedder.embed_documents(nf_texts)
    embed_s = time.perf_counter() - t0

    chunks = [
        Document(page_content=t, metadata={"id": cid})
        for t, cid in zip(nf_texts, nf_ids)
    ]
    t0 = time.perf_counter()
    store = FAISS.from_documents(
        chunks, embedding=_MixedEmbeddings(nf_texts, nf_vecs, embedder)
    )
    index_s = time.perf_counter() - t0

    # --- The local systems ---------------------------------------------------
    ce_model = CrossEncoder(CE_MODEL_NAME)
    rerank_retriever = _InlineRerankRetriever(
        store, embedder, ce_model, k_retrieve=WIDE_K, top_k=EVAL_K
    )
    colbert = _InlineColBERT(model_name=BGE_MODEL_NAME)

    subset_ids = set(nf_ids)
    covered = [
        (qid, q) for qid, q in nf_queries
        if qid in qrels and qrels[qid] & subset_ids
    ][:NF_MAX_QUERIES]

    t0 = time.perf_counter()
    baseline_scores, rr_scores, colbert_scores = [], [], []
    for qid, qtext in covered:
        gold = qrels[qid] & subset_ids
        wide = store.similarity_search_by_vector(embedder.embed_query(qtext), k=WIDE_K)
        baseline_scores.append(
            ndcg_at_k([d.metadata["id"] for d in wide[:EVAL_K]], gold, EVAL_K)
        )
        rr_scores.append(
            ndcg_at_k(
                [d.metadata["id"] for d in rerank_retriever.retrieve(qtext)],
                gold, EVAL_K,
            )
        )
        colbert_scores.append(
            ndcg_at_k(
                [d.metadata["id"] for d in colbert.rerank(qtext, wide, top_k=EVAL_K)],
                gold, EVAL_K,
            )
        )
    local_s = time.perf_counter() - t0

    def mean(scores: list[float]) -> float:
        return sum(scores) / len(scores) if scores else 0.0

    result = {
        "rows": covered,
        "baseline": mean(baseline_scores),
        "rerank_retriever": mean(rr_scores),
        "colbert": mean(colbert_scores),
        "indexed": len(nf_texts),
        "embed_s": embed_s,
        "index_s": index_s,
        "local_s": local_s,
        "optional": {},
    }

    # --- Classic LangChain wiring: ContextualCompressionRetriever + CE ------
    # The LangChain-native compression retriever, wired like the component
    # library's RerankRetriever but through langchain-classic. Its
    # CrossEncoderReranker needs a BaseCrossEncoder (score() -> list[float]);
    # the adapter above wraps the SAME sentence-transformers model. The
    # classic compressor strips metadata["score"], but nDCG needs only the
    # RANKING, so the measurement works here.
    from langchain_classic.retrievers import ContextualCompressionRetriever  # noqa: E402
    from langchain_classic.retrievers.document_compressors import (  # noqa: E402
        CrossEncoderReranker as ClassicCrossEncoderReranker,
    )
    classic_compressor = ClassicCrossEncoderReranker(
        model=_ScoringCrossEncoder(CE_MODEL_NAME), top_n=EVAL_K
    )
    classic_ccr = ContextualCompressionRetriever(
        base_compressor=classic_compressor,
        base_retriever=store.as_retriever(search_kwargs={"k": WIDE_K}),
    )
    classic_scores = []
    for qid, qtext in covered:
        gold = qrels[qid] & subset_ids
        ranked = classic_ccr.invoke(qtext)
        classic_scores.append(
            ndcg_at_k([d.metadata["id"] for d in ranked], gold, EVAL_K)
        )
    result["classic_ccr"] = mean(classic_scores)

    # --- MonoT5: only if the ~3GB checkpoint is already cached locally -----
    if MONT5_CACHE.exists():
        t5 = _InlineMonoT5()
        t5_scores = []
        for qid, qtext in covered:
            gold = qrels[qid] & subset_ids
            wide = store.similarity_search_by_vector(embedder.embed_query(qtext), k=WIDE_K)
            t5_scores.append(
                ndcg_at_k(
                    [d.metadata["id"] for d in t5.rerank(qtext, wide, top_k=EVAL_K)],
                    gold, EVAL_K,
                )
            )
        result["optional"]["monot5"] = {"status": "run", "ndcg": mean(t5_scores)}
    else:
        result["optional"]["monot5"] = {
            "status": "skip",
            "ndcg": None,
            "note": "checkpoint not in HF cache (~3GB download; run it yourself)",
        }

    # --- LLM pointwise: only with --with-llm and a GROQ key -----------------
    if with_llm:
        from dotenv import load_dotenv  # import on demand, like llms/groq.py
        load_dotenv()
    if with_llm and os.getenv("GROQ_API_KEY"):
        llm_rr = _InlineLLMPointwise(_GroqTextLLM(temperature=0.0))
        hits = 0
        shown = []
        for qid, qtext in covered[:LLM_POOL]:
            gold = qrels[qid] & subset_ids
            wide = store.similarity_search_by_vector(embedder.embed_query(qtext), k=WIDE_K)
            kept = llm_rr.rerank(qtext, wide, top_k=10)
            kept_ids = [d.metadata["id"] for d in kept]
            hit = bool(set(kept_ids) & gold)
            hits += 1 if hit else 0
            shown.append({"qid": qid, "kept_gold": hit})
        result["optional"]["llm"] = {
            "status": "run",
            "queries": LLM_POOL,
            "kept_gold": hits,
            "rows": shown,
        }
    else:
        result["optional"]["llm"] = {
            "status": "skip",
            "queries": LLM_POOL,
            "note": "pass --with-llm with GROQ_API_KEY set to run the LLM judge",
        }
    return result


## 5. Demo — print the artifact

`print_demo(exp)` prints the artifact: the mean nDCG@5 of each local system over the pool with its delta against the baseline — including the classic LangChain wiring's score as a cross-check — then the status of the two optional systems (ran vs. SKIP with the reason), and a takeaway on why every reranker lifts the baseline: each re-scores the 20 candidates with a model that sees the query, catching the fine-grained overlap a pooled cosine smears, at the cost of re-scoring every candidate.


In [ ]:
# --------------------------------------------------------------------------
# 5. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 04 — The repo's reranker arsenal, wired end-to-end")
    print(f"nfcorpus {exp['indexed']} docs, pool {len(exp['rows'])} queries, "
          f"nDCG@{EVAL_K}")
    print("=" * 66)

    print(f"\n[1] Local systems (mean nDCG@{EVAL_K} over {len(exp['rows'])} queries):")
    print(f"    baseline (bi top-{EVAL_K})              : {exp['baseline']:.4f}")
    print(f"    RerankRetriever (bi top-{WIDE_K} -> CE top-{EVAL_K})  : "
          f"{exp['rerank_retriever']:.4f} "
          f"({exp['rerank_retriever'] - exp['baseline']:+.4f})")
    print(f"    ColBERT MaxSim (BGE tokens, top-{WIDE_K} -> top-{EVAL_K}) : "
          f"{exp['colbert']:.4f} ({exp['colbert'] - exp['baseline']:+.4f})")
    print(f"    classic compression wiring (bi top-{WIDE_K} -> CE top-{EVAL_K}): "
          f"{exp['classic_ccr']:.4f} "
          f"({exp['classic_ccr'] - exp['baseline']:+.4f})")

    print(f"\n[2] Optional systems:")
    t5 = exp["optional"]["monot5"]
    print(f"    MonoT5 (pointwise seq2seq): "
          f"{'ran' if t5['status'] == 'run' else 'SKIP — ' + t5['note']}"
          + (f", nDCG@5 {t5['ndcg']:.4f}" if t5["status"] == "run" else ""))
    llm = exp["optional"]["llm"]
    if llm["status"] == "run":
        print(f"    LLM pointwise (Groq, {llm['queries']} queries x 10 candidates):")
        for row in llm["rows"]:
            print(f"      {row['qid']}: gold {'kept' if row['kept_gold'] else 'dropped'}")
    else:
        print(f"    LLM pointwise (Groq): SKIP — {llm['note']}")

    print(f"\n[3] Takeaway")
    print("    Every reranker lifts the baseline on the same pool, because")
    print("    each re-scores the 20 candidates with a model that sees the")
    print("    query: the cross-encoder reads (query, passage) pairs, and")
    print("    ColBERT matches query tokens against passage tokens (MaxSim)")
    print("    — catching the fine-grained overlap a pooled cosine smears.")
    print("    ColBERT reuses the SAME BGE embedder as retrieval, so the")
    print("    token-level precision costs no extra model download.")
    print("    The trade-off is compute: all of them re-score every")
    print("    candidate, which is why they only ever see a short list.")


## 6. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: exactly `NF_N_DOCS` docs indexed, a pool of at least 40 queries, every local nDCG in [0, 1], both local rerankers at or above the baseline, and both optional sections reporting a run-or-skip status. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 6. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    checks.append((f"exactly {NF_N_DOCS} nfcorpus docs indexed",
                   exp["indexed"] == NF_N_DOCS))
    checks.append((f"pool has {len(exp['rows'])} queries (>= 40)",
                   len(exp["rows"]) >= 40))

    checks.append(("baseline nDCG in [0, 1]",
                   0.0 <= exp["baseline"] <= 1.0))
    checks.append(("RerankRetriever nDCG in [0, 1]",
                   0.0 <= exp["rerank_retriever"] <= 1.0))
    checks.append(("ColBERT nDCG in [0, 1]",
                   0.0 <= exp["colbert"] <= 1.0))

    checks.append(("RerankRetriever >= baseline",
                   exp["rerank_retriever"] >= exp["baseline"]))
    checks.append(("ColBERT >= baseline",
                   exp["colbert"] >= exp["baseline"]))

    checks.append(("MonoT5 section reports run or skip",
                   exp["optional"]["monot5"]["status"] in ("run", "skip")))
    checks.append(("LLM section reports run or skip",
                   exp["optional"]["llm"]["status"] in ("run", "skip")))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Index nfcorpus once, then score the rerankers (RerankRetriever shape, ColBERT, plus the classic LangChain compression wiring) on the same query pool — a few minutes of CPU. `with_llm=False` (the lab's default): the optional Groq LLM pointwise reranker is skipped (set `GROQ_API_KEY` and run the lab with `--with-llm` to include it). `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

Pooled nDCG@5 per reranker on the same 40-query pool — who wins where, and how much the rerankers buy over the bi-encoder baseline.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the nfcorpus files are intact.


In [ ]:
verify_gate(exp)
